# Laboratorium 4: Probabilistyczna reprezentacja wiedzy

Sieć bayesowska platformy e-learningowej uczelni: budowa, wnioskowanie dokładne i przybliżone, d-separacja, uczenie parametrów i struktury z danych oraz porównanie ze współczynnikami pewności systemu MYCIN.

In [ ]:
# pgmpy nie jest preinstalowany w Colabie; pozostałe biblioteki są
%pip install -q pgmpy

In [ ]:
import itertools
import time
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

import pgmpy
# pgmpy 1.1 przy imporcie modułów inference i sampling wczytuje wewnętrznie przestarzały
# pakiet pgmpy.estimators, który zgłasza FutureWarning; filtr dotyczy tylko tego komunikatu
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=r".*pgmpy\.estimators\.StructureScore.*",
                            category=FutureWarning)
    from pgmpy.inference import VariableElimination
    from pgmpy.sampling import BayesianModelSampling
from pgmpy.base import DAG
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD, State
from pgmpy.example_models import load_model
from pgmpy.parameter_estimator import DiscreteMLE, DiscreteBayesianEstimator
from pgmpy.causal_discovery import HillClimbSearch, PC
from pgmpy.structure_score import BIC

RANDOM_STATE = 42
pd.set_option("display.precision", 4)
print("pgmpy", pgmpy.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

## Zadanie 1: Od reguł do niepewności

In [ ]:
# Dziennik monitoringu platformy: 2000 godzin, liczba godzin z danym układem zdarzeń
log_counts = pd.DataFrame(
    {"Alarm=tak": [38, 40], "Alarm=nie": [2, 1920]},
    index=["Awaria=tak", "Awaria=nie"],
)
log_counts

In [ ]:
# Policz z tabeli P(Awaria), P(Awaria | Alarm), P(Alarm | Awaria) i P(nie Awaria | nie Alarm).
# Porównaj je z regułą "JEŻELI Alarm TO Awaria" z systemu regułowego

In [ ]:
# Test diagnostyczny rzadkiej choroby
PREVALENCE = 0.001   # P(choroba)
SENSITIVITY = 0.99   # P(test dodatni | choroba)
SPECIFICITY = 0.95   # P(test ujemny | brak choroby)

In [ ]:
# Napisz funkcję posterior_positive(prior, sens, spec) z reguły Bayesa.
# Policz P(choroba | test dodatni) oraz wynik po drugim, warunkowo niezależnym teście dodatnim.
# Sprawdź wynik na częstościach naturalnych dla 100 000 osób

In [ ]:
# Narysuj P(choroba | test dodatni) w funkcji częstości choroby (skala logarytmiczna)
# dla kilku wartości swoistości: 0.90, 0.95, 0.99, 0.999

In [ ]:
# Łańcuch Przeciazenie -> Awaria -> Alarm (stan 0 = "nie", 1 = "tak")
p_overload = np.array([0.90, 0.10])                    # P(Przeciazenie)
p_failure_given_overload = np.array([[0.98, 0.02],     # P(Awaria | Przeciazenie=nie)
                                     [0.40, 0.60]])    # P(Awaria | Przeciazenie=tak)
p_alarm_given_failure = np.array([[0.98, 0.02],        # P(Alarm | Awaria=nie)
                                  [0.05, 0.95]])       # P(Alarm | Awaria=tak)

In [ ]:
# Zbuduj rozkład łączny J[p, a, l] (np. np.einsum) i pokaż go jako tabelę 8 wierszy; sprawdź, że sumuje się do 1

In [ ]:
# Sprawdź, że Alarm zależy od Przeciazenia brzegowo, ale jest od niego niezależny warunkowo przy znanej Awarii:
# porównaj P(Alarm | Przeciazenie) z P(Alarm) oraz P(Alarm | Awaria, Przeciazenie) z P(Alarm | Awaria)

In [ ]:
# Tabela: liczba niezależnych parametrów rozkładu łącznego n zmiennych binarnych (2^n - 1)
# i górne ograniczenie dla sieci, w której każdy węzeł ma najwyżej 2 rodziców, dla n = 3, 7, 20, 37, 100

### Wnioski do zadania 1

(uzupełnij)

## Zadanie 2: Sieć bayesowska platformy e-learningowej

In [ ]:
STATES = ["nie", "tak"]
PARENTS = {
    "Aktualizacja": [],                    # w ciągu doby wdrożono aktualizację platformy
    "Sesja": [],                           # trwa sesja egzaminacyjna
    "Blad": ["Aktualizacja"],              # błąd w oprogramowaniu platformy
    "Przeciazenie": ["Sesja"],             # przeciążenie serwerów
    "Awaria": ["Blad", "Przeciazenie"],    # platforma niedostępna
    "Alarm": ["Awaria"],                   # alarm z systemu monitoringu
    "Zgloszenia": ["Awaria", "Sesja"],     # lawina zgłoszeń studentów do helpdesku
}
# P(zmienna = "tak" | rodzice); konfiguracje rodziców w kolejności itertools.product(STATES, ...),
# czyli dla dwóch rodziców: (nie, nie), (nie, tak), (tak, nie), (tak, tak)
P_TAK = {
    "Aktualizacja": [0.10],
    "Sesja": [0.20],
    "Blad": [0.02, 0.30],
    "Przeciazenie": [0.05, 0.40],
    "Awaria": [0.01, 0.60, 0.70, 0.95],
    "Alarm": [0.02, 0.95],
    "Zgloszenia": [0.02, 0.10, 0.70, 0.95],
}


def cpd_table(cpd):
    # CPD jako DataFrame: wiersze to konfiguracje rodziców, kolumny to stany zmiennej
    parents = cpd.variables[1:]
    if parents:
        index = pd.MultiIndex.from_tuples(
            list(itertools.product(*[cpd.state_names[p] for p in parents])), names=parents)
    else:
        index = ["(brak rodziców)"]
    columns = [f"{cpd.variable}={s}" for s in cpd.state_names[cpd.variable]]
    return pd.DataFrame(cpd.get_values().T, index=index, columns=columns)

In [ ]:
# Napisz funkcję make_cpd(var), która z PARENTS i P_TAK tworzy TabularCPD (state_names = STATES).
# Zbuduj DiscreteBayesianNetwork, dodaj wszystkie CPD, wywołaj check_model() i pokaż cpd_table dla Awarii

In [ ]:
# Zepsuj celowo jedną kolumnę CPD Alarmu (kolumna nie sumuje się do 1) w kopii modelu
# i sprawdź komunikat check_model()

In [ ]:
# Policz liczbę niezależnych parametrów każdej CPD i całej sieci; porównaj z 2^7 - 1.
# Policz prawdopodobieństwo pełnego stanu (np. Sesja, Przeciazenie, Awaria, Alarm, Zgloszenia = tak, reszta nie)
# jako iloczyn wpisów CPD i porównaj z model.get_state_probability

In [ ]:
# Narysuj sieć (networkx, stałe pozycje węzłów), przy węzłach podaj liczbę parametrów.
# Wypisz koc Markowa Awarii (get_markov_blanket) i lokalne niezależności (local_independencies)

### Wnioski do zadania 2

(uzupełnij)

## Zadanie 3: Wnioskowanie dokładne i d-separacja

In [ ]:
# Utwórz VariableElimination(model) i funkcję p_tak(var, evidence) zwracającą P(var = tak | evidence).
# Zbuduj tabelę zapytań: a priori, przyczynowe (od przyczyny do skutku), diagnostyczne (od skutku do przyczyny),
# międzyprzyczynowe (Blad przy Awaria=tak, także z Przeciazenie=tak/nie albo Sesja=tak) i mieszane

In [ ]:
# Przy Alarm=tak wypisz rozkład łączny a posteriori (Sesja, Przeciazenie), wynik map_query dla tej pary
# i stany wybrane osobno z rozkładów brzegowych. Dodaj MPE: map_query po wszystkich nieobserwowanych zmiennych

In [ ]:
# Dla listy trójek (X, Y, Z) sprawdź is_dconnected(X, Y, observed=Z) i numerycznie największą różnicę
# |P(X=tak | Z) - P(X=tak | Y, Z)| po wszystkich stanach Y i Z. Uwzględnij co najmniej:
# (Aktualizacja, Sesja) bez obserwacji, przy Awarii, przy Alarmie i przy Zgłoszeniach; (Blad, Alarm | Awaria);
# (Przeciazenie, Zgloszenia | Awaria) oraz (Przeciazenie, Zgloszenia | Awaria, Sesja)

In [ ]:
# Wypisz active_trail_nodes dla Aktualizacji bez obserwacji i przy obserwowanym Alarmie
# oraz minimal_dseparator dla pary (Blad, Zgloszenia)

### Wnioski do zadania 3

(uzupełnij)

## Zadanie 4: Wnioskowanie przybliżone w sieci ALARM

In [ ]:
# Wczytaj sieć ALARM: load_model("bnlearn/alarm"). Podaj liczbę węzłów, krawędzi, największą liczbę rodziców,
# liczbę niezależnych parametrów CPD i rząd wielkości liczby stanów rozkładu łącznego

In [ ]:
EVIDENCE = {"CVP": "HIGH", "BP": "LOW", "HISTORY": "TRUE"}
TARGET, TARGET_STATE = "LVFAILURE", "TRUE"

In [ ]:
# Policz dokładnie P(LVFAILURE=TRUE | EVIDENCE) metodą eliminacji zmiennych i zmierz czas.
# Policz też prawdopodobieństwo samych obserwacji P(EVIDENCE) (zapytanie łączne o zmienne z EVIDENCE)

In [ ]:
# Dla n w SIZES = [100, 300, 1000, 3000, 10000, 30000] i 5 ziaren (RANDOM_STATE + r) porównaj:
#  - próbkowanie z odrzucaniem: alarm.simulate(n, seed=...), zostaw wiersze zgodne z EVIDENCE, oszacuj P(TARGET)
#  - ważenie wiarygodnością: BayesianModelSampling(alarm).likelihood_weighted_sample(evidence=[State(...)], size=n, seed=...),
#    estymator to średnia ważona kolumną "_weight"
# Zapisz liczbę zachowanych próbek, błąd bezwzględny względem wyniku dokładnego i czas; zbuduj tabelę średnich po n

In [ ]:
# Narysuj średni błąd obu metod w funkcji n (skale logarytmiczne, linia odniesienia ~ 1/sqrt(n))
# oraz średnią liczbę próbek zachowanych przy odrzucaniu na tle n * P(EVIDENCE)

### Wnioski do zadania 4

(uzupełnij)

## Zadanie 5: Uczenie parametrów i struktury

In [ ]:
# Dla n = 50, 200, 1000, 5000, 20000 i 5 ziaren wygeneruj dane model.simulate(n, seed=...),
# naucz parametry na prawdziwej strukturze: DiscreteMLE i DiscreteBayesianEstimator(prior_type="BDeu", equivalent_sample_size=5),
# w obu przypadkach state_names=model.states. Porównaj z prawdziwymi CPD:
#  - średnia odległość całkowitej zmienności między kolumnami CPD,
#  - dywergencja KL(P_prawdziwy || P_nauczony) rozkładów łącznych (128 stanów, zapytanie łączne o wszystkie zmienne)

In [ ]:
# Dla jednej próbki n = 50 pokaż tabelę liczności (Blad, Przeciazenie) x Awaria
# i zestaw CPD Awarii: prawdziwą, z MLE i z BDeu. Co dzieje się z konfiguracjami rodziców, których nie ma w danych?

Do uczenia struktury służą dwie metody: przeszukiwanie przestrzeni grafów z oceną BIC (hill climbing) i metoda PC oparta na testach niezależności warunkowej. Funkcja `hill_climb` w komórce poniżej jest prostą implementacją przeszukiwania zachłannego. `HillClimbSearch` z pgmpy działa tak samo, ale remisy między ruchami rozstrzyga w kolejności zależnej od haszowania napisów, więc po restarcie środowiska może zwrócić inny graf. Własna wersja daje wynik powtarzalny.

In [ ]:
def edge_marks(pdag):
    # krawędź szkieletu -> kierunek (para) albo "-" dla krawędzi nieskierowanej
    marks = {frozenset(e): tuple(e) for e in pdag.directed_edges}
    marks.update({frozenset(e): "-" for e in pdag.undirected_edges})
    return marks


def compare_with_cpdag(true_pdag, est_pdag):
    t, e = edge_marks(true_pdag), edge_marks(est_pdag)
    common = set(t) & set(e)
    return {"krawędzie": len(e), "TP": len(common), "FP": len(set(e) - set(t)), "FN": len(set(t) - set(e)),
            "zgodne oznaczenie": sum(t[k] == e[k] for k in common)}


def consistent_dag(pdag):
    # DAG z tej samej klasy co PDAG albo None, gdy PDAG nie ma spójnego rozszerzenia
    # (to_dag() zwraca wtedy graf z cyklem albo z nową v-strukturą)
    dag = pdag.to_dag()
    if not nx.is_directed_acyclic_graph(dag) or edge_marks(dag.to_pdag()) != edge_marks(pdag):
        return None
    return dag


def hill_climb(data, score, max_parents=3, eps=1e-6):
    # zachłanne przeszukiwanie: dodanie, usunięcie albo odwrócenie jednej krawędzi;
    # remisy rozstrzyga stała kolejność kolumn, więc wynik jest powtarzalny
    nodes = list(data.columns)
    g = nx.DiGraph()
    g.add_nodes_from(nodes)
    cache = {}

    def local(v, parents):
        key = (v, tuple(sorted(parents)))
        if key not in cache:
            cache[key] = score.local_score(v, key[1])
        return cache[key]

    while True:
        best_delta, best_op = eps, None
        for x, y in itertools.permutations(nodes, 2):
            py, px = list(g.predecessors(y)), list(g.predecessors(x))
            ops = []
            if g.has_edge(x, y):
                without = [p for p in py if p != x]
                ops.append((local(y, without) - local(y, py), ("-", x, y)))
                g.remove_edge(x, y)
                reversible = not nx.has_path(g, x, y) and len(px) < max_parents
                g.add_edge(x, y)
                if reversible:
                    delta = local(y, without) - local(y, py) + local(x, px + [y]) - local(x, px)
                    ops.append((delta, ("odwróć", x, y)))
            elif not g.has_edge(y, x) and len(py) < max_parents and not nx.has_path(g, y, x):
                ops.append((local(y, py + [x]) - local(y, py), ("+", x, y)))
            for delta, op in ops:
                if delta > best_delta:
                    best_delta, best_op = delta, op
        if best_op is None:
            dag = DAG()
            dag.add_nodes_from(nodes)
            dag.add_edges_from(g.edges())
            return dag
        kind, x, y = best_op
        if kind in ("-", "odwróć"):
            g.remove_edge(x, y)
        if kind == "odwróć":
            g.add_edge(y, x)
        if kind == "+":
            g.add_edge(x, y)

In [ ]:
# Prawdziwą klasę równoważności Markowa daje model.to_pdag(). Dla n = 500, 2000, 10000 (seed=RANDOM_STATE):
#  - hill_climb(data, BIC(data)) z komórki powyżej (zwraca DAG),
#  - PC(ci_test="chi_square", significance_level=0.01, n_jobs=1, show_progress=False).fit(data).causal_graph_ (zwraca PDAG)
# Porównaj oba wyniki z prawdziwym CPDAG funkcją compare_with_cpdag (DAG zamień na PDAG metodą to_pdag()).
# Dodaj BIC nauczonej struktury i prawdziwej sieci (BIC(data).score(...)). PDAG z PC zamień na DAG funkcją
# consistent_dag; gdy zwróci None, wpisz NaN

In [ ]:
# Dla n = 10000 wypisz, którymi krawędziami wynik HillClimb różni się od prawdziwego CPDAG.
# Odwróć w prawdziwej sieci krawędź Aktualizacja -> Blad, a osobno Blad -> Awaria; dla obu wariantów sprawdź
# is_iequivalent względem prawdziwej sieci i porównaj BIC na tych samych danych

In [ ]:
# Uruchom HillClimbSearch(scoring_method="bic-d", return_type="dag", show_progress=False) z pgmpy na tych samych
# danych (n = 10000) i porównaj wynik z własnym hill_climb (krawędzie, BIC, klasa równoważności).
# Uruchom komórkę kilka razy po restarcie środowiska: czy wynik pgmpy jest za każdym razem taki sam?

### Wnioski do zadania 5

(uzupełnij)

## Zadanie 6: Współczynniki pewności MYCIN

In [ ]:
# Zaimplementuj cf_combine(a, b) (trzy przypadki MYCIN), cf_chain(cf_rule, cf_premise),
# cf_from_prob(post, prior) z definicji CF = MB - MD oraz odwrotność prob_from_cf(cf, prior).
# Sprawdź na przykładzie, że kolejność łączenia trzech CF nie zmienia wyniku

In [ ]:
# Z sieci z zadania 2 wyznacz CF czterech reguł z definicji MYCIN (cf_from_prob, a priori z p_tak):
#   R1: Alarm -> Awaria, R2: Zgloszenia -> Awaria, R3: Awaria -> Blad, R4: Przeciazenie -> Blad.
# Zbuduj tabelę porównującą CF (i odpowiadające mu prawdopodobieństwo) z reguł oraz z sieci dla przypadków:
#  1) Awaria przy Alarm i Zgloszenia (R1 i R2 równolegle), 2) Blad przy Alarm (łańcuch R1, R3),
#  3) Blad przy Awaria i Przeciazenie (R3 i R4), 4) Awaria, gdy ten sam alarm przychodzi dwoma kanałami

### Wnioski do zadania 6

(uzupełnij)

## Zadanie 7: Pytania kontrolne

1. Dlaczego reguła "JEŻELI Alarm TO Awaria" i liczba P(Awaria | Alarm) nie przenoszą tej samej wiedzy? Czego reguła nie mówi?
2. Ile niezależnych parametrów ma rozkład łączny siedmiu zmiennych binarnych, a ile sieć z zadania 2? Z czego wynika różnica?
3. Na czym polega efekt wyjaśniania (explaining away) i dlaczego obserwacja wspólnego skutku uzależnia od siebie niezależne przyczyny?
4. Dlaczego próbkowanie z odrzucaniem zawodzi przy mało prawdopodobnych obserwacjach i co zmienia ważenie wiarygodnością?
5. Dlaczego z samych danych obserwacyjnych nie da się ustalić kierunku każdej krawędzi sieci?
6. Kiedy współczynniki pewności dają wynik niezgodny z rachunkiem prawdopodobieństwa i co to mówi o modularności reguł?

### Odpowiedzi

(uzupełnij)